In [1]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets

def pruning_report(model):
    print("\n=== PRUNING REPORT ===\n")
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            w = module.weight.detach().cpu()

            # A filter is pruned if ALL its weights are zero
            filter_sums = w.abs().sum(dim=(1,2,3))
            pruned = (filter_sums == 0).sum().item()
            total = w.shape[0]

            print(f"{name:<40}  {pruned:>3}/{total:<3} filters pruned")

    print("\n=======================\n")

# Load model
# model = torch.load("../model/prunned_fruit_mobilenetv2.pth")
model = torch.load("fruit_mobilenetv2.pth")
# show report
pruning_report(model)



=== PRUNING REPORT ===

features.0.0                                0/32  filters pruned
features.1.conv.0.0                         0/32  filters pruned
features.1.conv.1                           0/16  filters pruned
features.2.conv.0.0                         0/96  filters pruned
features.2.conv.1.0                         0/96  filters pruned
features.2.conv.2                           0/24  filters pruned
features.3.conv.0.0                         0/144 filters pruned
features.3.conv.1.0                         0/144 filters pruned
features.3.conv.2                           0/24  filters pruned
features.4.conv.0.0                         0/144 filters pruned
features.4.conv.1.0                         0/144 filters pruned
features.4.conv.2                           0/32  filters pruned
features.5.conv.0.0                         0/192 filters pruned
features.5.conv.1.0                         0/192 filters pruned
features.5.conv.2                           0/32  filters pruned


In [2]:
# preprocess images
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

#  Validation
model.eval()
val_correct, val_total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100*val_correct/val_total:.2f}%")

Validation Accuracy: 89.35%


In [4]:
for m in model.modules():
    if isinstance(m, torch.nn.BatchNorm2d):
        print("mean:", m.running_mean[:5])
        print("var :", m.running_var[:5])
        break

mean: tensor([-0.0324,  0.0105,  0.0386, -0.0060, -0.4758])
var : tensor([0.1520, 0.0621, 0.6983, 0.0620, 6.2794])
